# 2.5 · 置信区间 / Confidence Intervals

> **课程定位**
> 2.3 给了 SE，本课把它装进**区间**：$\bar{x} \pm t \cdot \mathrm{SE}$。重头戏有三：**"95% 置信"的真实含义（多数人答错）、z vs t 的选择、比例 CI 的 Wald 陷阱**。
> SE becomes an interval. Three centerpieces: what "95% confident" really means (most get it wrong), z vs t, and the Wald trap for proportions.

> 💡 **面试相关**
> - "解释 95% 置信区间" ★★★★★（标准答案与错误答案一字之差）
> - "什么时候用 t 不用 z" ★★★★
> - "比例的置信区间怎么算" ★★★★（A/B 测试直接用）
> - "CI 和假设检验的关系" ★★★

---

## 目录
1. [⭐ "95% 置信"到底什么意思（覆盖率模拟）](#1)
2. [均值 CI：z 还是 t](#2)
3. [t 分布：为什么尾巴更厚](#3)
4. [比例 CI：Wald 的失败与 Wilson 的修正 ⭐](#4)
5. [两组差的 CI（A/B 测试的雏形）](#5)
6. [方差与中位数的 CI](#6)
7. [⚠ 五大误读](#7)
8. [实战：覆盖率大检验](#8)
9. [小结](#9)


<a id="1"></a>
## 1. ⭐ "95% 置信"到底什么意思 / What 95% Actually Means

$$\mathrm{CI}_{95\%} = \bar{x} \pm 1.96 \cdot \frac{s}{\sqrt{n}}$$

**❌ 错误理解**（最常见）："真值有 95% 的概率落在这个区间里。"
**✅ 正确理解**："**这套造区间的程序**，长期使用时 95% 的次数会罩住真值。"

差别在哪：频率学派里 $\mu$ 是**常数**不是随机变量——某个具体区间要么含 $\mu$（概率 1）要么不含（概率 0）。**随机的是区间**（每次抽样都变），不是真值。"95%" 是**程序的质保书**，不是单个区间的属性。
The parameter is fixed; the interval is random. 95% is a warranty on the *procedure*, not a probability statement about one realized interval. (Bayesian credible intervals — lesson 2.10 — do support the intuitive reading.)

**眼见为实**——画 100 个区间：


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

MU, SIGMA, n = 50.0, 10.0, 30

fig, ax = plt.subplots(figsize=(10, 4.5))
n_hit = 0
for i in range(100):
    x = rng.normal(MU, SIGMA, n)
    se = x.std(ddof=1) / np.sqrt(n)
    t_crit = st.t.ppf(0.975, df=n-1)
    lo, hi = x.mean() - t_crit*se, x.mean() + t_crit*se
    hit = lo <= MU <= hi
    n_hit += hit
    ax.plot([lo, hi], [i, i], color="steelblue" if hit else "red",
            lw=1.6 if hit else 2.4, alpha=0.8)
ax.axvline(MU, color="black", ls="--", lw=1.5, label=f"true μ = {MU}")
ax.set_title(f"100 random 95% CIs — {n_hit} cover the truth, {100-n_hit} (red) miss")
ax.legend(); ax.set_yticks([])
plt.tight_layout(); plt.show()


**约 95 条蓝（罩住）、5 条红（漏掉）**——而且**事先无法知道自己手里这条是不是红的**。这就是"95%"的全部含义。
~95 blue, ~5 red — and you can never know which color yours is. That's the entire meaning.


<a id="2"></a>
## 2. 均值 CI：z 还是 t / z or t

| 已知什么 | 用什么 | 区间 |
|---|---|---|
| $\sigma$ 已知（几乎不可能）| z | $\bar{x} \pm z_{\alpha/2}\,\sigma/\sqrt{n}$ |
| $\sigma$ 未知（**现实**）| **t** ⭐ | $\bar{x} \pm t_{\alpha/2,\,n-1}\,s/\sqrt{n}$ |

**为什么 t**：用 $s$ 替代 $\sigma$ 引入了**额外不确定性**（$s$ 本身在抖）。t 分布的厚尾恰好补偿这份"对 $s$ 的不自信"。

实用规则：**永远用 t**——$n$ 大时 t 自动趋近 z（$t_{0.975, 1000} = 1.962 \approx 1.96$），没有任何损失。
Just always use t: it converges to z anyway for large n.


In [ ]:
# t 临界值随 df 收敛到 1.96 / t critical values converge to z
print(f"{'n':>6} {'t_crit (95%)':>13}    z_crit = 1.960")
for n_ in [3, 5, 10, 30, 100, 1000]:
    print(f"{n_:>6} {st.t.ppf(0.975, n_-1):>13.3f}")
print("\nn=3 时 t 临界值 4.30 — 比 z 宽一倍多；小样本误用 z = 区间假窄、覆盖不足")


<a id="3"></a>
## 3. t 分布：为什么尾巴更厚 / Why t Has Fat Tails

定义：$Z \sim \mathcal{N}(0,1)$，$V \sim \chi^2_\nu$ 独立时
$$T = \frac{Z}{\sqrt{V/\nu}} \sim t_\nu$$

样本场景的对应：$\dfrac{\bar{x} - \mu}{s/\sqrt{n}} \sim t_{n-1}$（**正态母体下精确成立**，Gosset/“Student” 1908 在吉尼斯啤酒厂推出）。

分母里 $s$ 偶尔**偏小**（碰巧抽到集中样本）→ 比值被放大 → 厚尾。$\nu \to \infty$ 时 $s \to \sigma$，t → 正态。
Occasionally $s$ comes out small by chance, inflating the ratio — hence fat tails. As df grows, $s$ pins down and t collapses to normal.


In [ ]:
xs = np.linspace(-5, 5, 400)
fig, ax = plt.subplots(figsize=(9, 3.6))
for df, c in [(2, "C3"), (5, "C1"), (30, "C2")]:
    ax.plot(xs, st.t.pdf(xs, df), c, lw=1.8, label=f"t(df={df})")
ax.plot(xs, st.norm.pdf(xs), "k--", lw=2, label="N(0,1)")
ax.set_yscale("log"); ax.set_ylim(1e-5, 1)            # log 轴专看尾部 / log scale exposes tails
ax.legend(); ax.set_title("t vs normal — log scale shows the tail gap")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 比例 CI：Wald 的失败与 Wilson 的修正 ⭐ / Proportions

转化率、点击率、留存率——**DS 最常算的就是比例 CI**。

**Wald 区间**（教科书默认，**实际很烂**）：
$$\hat{p} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

问题：$\hat p$ 接近 0 或 1 时（低转化率场景！）覆盖率塌方，还会越界（负概率）。

**Wilson 区间**（工业推荐 ⭐）：解二次方程而非插值，中心点被"拉向 0.5"：
$$\frac{\hat{p} + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \;\pm\; \frac{z}{1 + \frac{z^2}{n}}\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}$$

**极端案例**：$n=50$ 全部成功（$\hat p = 1$）→ Wald 给 $[1, 1]$（宣称零不确定性，荒谬）；Wilson 给 $[0.93, 1.0]$（合理）。


In [ ]:
from statsmodels.stats.proportion import proportion_confint

# 覆盖率对比: 真 p=0.05 (典型转化率), n=100
# Coverage shoot-out at p=0.05, n=100
p_true, n, n_sim = 0.05, 100, 30_000
cover = {"normal": 0, "wilson": 0, "beta": 0}     # beta = Clopper-Pearson (精确保守)
for _ in range(n_sim):
    k = rng.binomial(n, p_true)
    for method in cover:
        lo, hi = proportion_confint(k, n, alpha=0.05, method=method)
        cover[method] += (lo <= p_true <= hi)

print(f"真 p = {p_true}, n = {n}, 名义覆盖率 95%")
for m, c in cover.items():
    flag = "✓" if abs(c/n_sim - 0.95) < 0.01 else ("⚠ 不足!" if c/n_sim < 0.94 else "(保守)")
    print(f"  {m:<8}: 实际覆盖 {c/n_sim:.1%}  {flag}")

# 极端案例 / The degenerate case
print(f"\nn=50 全成功: Wald   = {proportion_confint(50, 50, method='normal')}")
print(f"            Wilson = {tuple(round(v,3) for v in proportion_confint(50, 50, method='wilson'))}")


**Wald 实际覆盖 ~88-92%（宣称 95%）**——低转化率 A/B 测试用 Wald 等于系统性过度自信。**结论：比例永远用 Wilson**（`statsmodels` 一个参数的事）。
Wald under-covers at exactly the low-p regime where conversion metrics live. Always Wilson.


<a id="5"></a>
## 5. 两组差的 CI / CI for a Difference — A/B 测试雏形

实验组 vs 对照组的均值差：
$$(\bar{x}_A - \bar{x}_B) \pm t^* \sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}$$

（Welch 形式——**不假设两组方差相等**，自由度用 Welch–Satterthwaite 近似。2.6 节详述为什么默认 Welch。）

**判读规则 ⭐**：差值 CI **含 0** ⇔ "差异不显著"（与 α=0.05 双侧检验等价）。**报告 CI 优于只报 p 值**——它同时告诉你方向、大小、不确定性。
If the CI for the difference contains 0, the difference isn't significant at the matching α. Reporting the CI beats reporting just p — it shows direction, magnitude, and uncertainty at once.


In [ ]:
# 模拟 A/B: 新按钮提升停留时长? / Did the new button lift dwell time?
old = rng.normal(120, 30, 800)        # 对照
new = rng.normal(124, 30, 800)        # 实验 (真提升 +4)

diff = new.mean() - old.mean()
se = np.sqrt(new.var(ddof=1)/len(new) + old.var(ddof=1)/len(old))
df_w = se**4 / ((new.var(ddof=1)/len(new))**2/(len(new)-1) + (old.var(ddof=1)/len(old))**2/(len(old)-1))
t_crit = st.t.ppf(0.975, df_w)

print(f"差值 = {diff:+.2f} 秒,  95% CI = [{diff - t_crit*se:.2f}, {diff + t_crit*se:.2f}]")
print(f"CI 不含 0 → 显著提升; 且区间宽度告诉你: 真提升可能在 1~7 秒之间")


<a id="6"></a>
## 6. 方差与中位数的 CI / CI for Variance & Median

**方差**（正态母体下，基于 $\chi^2$，**不对称**）：
$$\Big[\frac{(n-1)s^2}{\chi^2_{\alpha/2,\,n-1}},\; \frac{(n-1)s^2}{\chi^2_{1-\alpha/2,\,n-1}}\Big]$$

⚠ 这条公式**对非正态极度敏感**（不像均值 CI 有 CLT 兜底）——实战中方差的 CI 用 bootstrap（2.11）。

**中位数**（分布无关！基于顺序统计量 + 二项分布）：取排序样本的第 $j$ 到第 $k$ 个值，使 $\Pr(j \le \#\{x_i < m\} < k)$ ≈ 0.95。**完全不依赖分布形状**——重尾数据的安全选择。
The median CI from order statistics is fully distribution-free — the safe choice for heavy tails.


In [ ]:
# 中位数的分布无关 CI / Distribution-free median CI
x = np.sort(rng.lognormal(1, 0.8, 199))         # 重偏态数据 / heavily skewed
n_ = len(x)
# 二项分布找顺序统计量的秩 / binomial ranks
j = st.binom.ppf(0.025, n_, 0.5).astype(int)
k = st.binom.ppf(0.975, n_, 0.5).astype(int) + 1
print(f"样本中位数 = {np.median(x):.3f}")
print(f"95% CI (order-statistic) = [{x[j]:.3f}, {x[k]:.3f}]   ← 无任何分布假设")
print(f"真中位数 = e^1 = {np.e:.3f}  (在区间内 ✓)")


<a id="7"></a>
## 7. ⚠ 五大误读 / Five Misreadings

| # | 误读 | 纠正 |
|---|---|---|
| 1 | "真值有 95% 概率在区间内" | 区间随机，真值固定；95% 是程序长期成绩 |
| 2 | "区间内的值等可能" | CI 不是分布；中心附近的值与数据更相容而已 |
| 3 | "两组 CI 重叠 = 不显著" ⭐ | **错！** 单组 CI 可重叠而**差值** CI 不含 0（见下面演示）；判断显著看差值 CI |
| 4 | "n 大 → CI 窄 → 效应重要" | 窄区间只说明估得准；效应可以又准又小到没用（统计显著 ≠ 业务显著）|
| 5 | "95% 的数据落在 CI 内" | 那是预测区间 / 参考范围（宽 $\sqrt{n}$ 倍），不是均值 CI |


In [ ]:
# 误读 3 现场演示: 单组 CI 重叠但差异显著
# Overlapping individual CIs, yet significant difference
a = rng.normal(100, 15, 200)
b = rng.normal(104, 15, 200)

def mean_ci(x):
    se = x.std(ddof=1)/np.sqrt(len(x))
    t_ = st.t.ppf(0.975, len(x)-1)
    return x.mean()-t_*se, x.mean()+t_*se

ci_a, ci_b = mean_ci(a), mean_ci(b)
d = b.mean() - a.mean()
se_d = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
ci_d = (d - 1.96*se_d, d + 1.96*se_d)

print(f"A 的 CI: [{ci_a[0]:.1f}, {ci_a[1]:.1f}]")
print(f"B 的 CI: [{ci_b[0]:.1f}, {ci_b[1]:.1f}]   ← 两区间重叠")
print(f"差值 CI: [{ci_d[0]:.2f}, {ci_d[1]:.2f}]   ← 不含 0, 差异显著!")
print("\n原理: 单组 CI 重叠检验隐含 SE_A+SE_B, 但正确的 SE_diff=√(SE_A²+SE_B²) 更小")


<a id="8"></a>
## 8. 实战：覆盖率大检验 / The Coverage Audit

**覆盖率模拟是检验任何 CI 方法的金标准**：造数据 → 造区间 1 万次 → 数罩住真值的比例。把本课方法对**偏态数据**统一审计：
Coverage simulation is the gold standard for auditing any CI recipe.


In [ ]:
# 对 lognormal(偏态)数据审计三种均值 CI / Audit mean CIs on skewed data
true_mean = np.exp(0.5)                  # lognormal(0,1) 的均值 = e^0.5
n_sim = 20_000

def audit(n):
    t_cover = boot_cover = 0
    t_crit = st.t.ppf(0.975, n-1)
    for _ in range(n_sim // 10):          # bootstrap 较慢, 用 2000 次
        x = rng.lognormal(0, 1, n)
        se = x.std(ddof=1)/np.sqrt(n)
        t_cover += (x.mean()-t_crit*se <= true_mean <= x.mean()+t_crit*se)
        # percentile bootstrap (预览 2.11) / percentile bootstrap preview
        bm = np.array([rng.choice(x, n).mean() for _ in range(400)])
        lo, hi = np.percentile(bm, [2.5, 97.5])
        boot_cover += (lo <= true_mean <= hi)
    return t_cover/(n_sim//10), boot_cover/(n_sim//10)

print(f"lognormal 数据 (skew≈6), 名义 95%:")
print(f"{'n':>6} {'t 区间':>9} {'bootstrap':>10}")
for n_ in [15, 50, 200]:
    t_c, b_c = audit(n_)
    print(f"{n_:>6} {t_c:>9.1%} {b_c:>10.1%}")


**重偏态 + 小样本下，t 区间和 percentile bootstrap 都覆盖不足**（~88-92%）——这正是 2.3 节"偏态要更大 n"的置信区间版。n=200 后 t 区间靠 CLT 恢复。

**审计的元教训**：任何 CI 配方都该先在"和你数据形状像"的模拟数据上跑覆盖率，再上生产。2.11 节的 BCa bootstrap 就是为这种场景设计的改良。
The meta-lesson: audit any CI recipe on simulated data shaped like yours before production. BCa bootstrap (2.11) is the upgrade built for this.


<a id="9"></a>
## 9. 小结 / Summary

```
CI = 点估计 ± 临界值 × SE
  ├── 含义: 程序的长期覆盖率, 不是单区间的概率 ⭐
  ├── 均值: 永远用 t (大 n 自动 = z)
  ├── 比例: 永远用 Wilson (Wald 在低 p 塌方) ⭐
  ├── 两组差: Welch SE; 差值 CI 含 0 ⇔ 不显著
  ├── 中位数: 顺序统计量法, 分布无关
  └── 审计: 覆盖率模拟是金标准
```

### 💡 面试速查
1. **标准答案**："95% 指这套**程序**长期 95% 次数罩住真值；单个区间非蓝即红，只是你不知道"
2. **t vs z**：σ 未知用 t；厚尾补偿对 s 的不确定
3. **比例用 Wilson**：Wald 在 p≈0.05、全成功等场景覆盖塌方
4. **CI 重叠 ≠ 不显著**：判断显著看**差值** CI（SE 合成是平方和开根）
5. **统计显著 ≠ 业务显著**：窄区间 + 小效应很常见

### 下一节
**2.6 假设检验**——CI 的孪生兄弟：t 检验、卡方、ANOVA、非参，以及 p 值的正确打开方式。
